# CROSSWATCH — verified local research walkthrough

Run the comprehensive historical pipeline first with all three private FRED CSVs and a truthful local permission declaration (including the instructor-held approval mode). The notebook reads a **fingerprinted run manifest** and refuses stale or mismatched results. The synthetic QA release is not historical financial evidence.


In [ ]:
import os
import sys
from pathlib import Path
sys.path.insert(0, str(Path('Sources').resolve()))
from release_integrity import read_manifest
from historical_cases import load_cases, select_examples, matching_rules, case_figure
import pandas as pd

results_dir = Path(os.getenv('CROSSWATCH_RESULTS', 'private-results'))
raw_dir = Path(os.getenv('CROSSWATCH_RAW', 'private-inputs'))
manifest = read_manifest(results_dir, raw_dir=raw_dir)
print('Verified run ID:', manifest['run_id'])
print('Input kind:', manifest['run_kind'], '(synthetic QA is not historical market evidence)')
variant, split = 'five_features', 'test'
frame, metadata, rules, closes = load_cases(results_dir, raw_dir, variant, split)
examples = select_examples(frame)
frame.loc[list(examples.values()), ['anomaly_severity', 'IFOREST_ALERT', 'REGIME_ALERT', 'cross_asset_confirmation', 'forward_proxy']].assign(case=list(examples))


## Why VIX context changes an alert

Both forest alerting rules use the same severity score but different thresholds. The final chart displays the **subsequent actual or synthetic (depending on run kind)** equity path for research evaluation; it is not a prediction.


In [ ]:
import matplotlib.pyplot as plt
date = examples.get('Global forest only', next(iter(examples.values())))
fig = case_figure(frame, metadata, closes, date)
plt.show()
matching_rules(frame.loc[date], rules, split)


## Cross-method evaluation from the exact same run

The comparison tables are generated using the same partition and run manifest as the interactive dashboard. The equal-budget scores are retrospective.


In [ ]:
operating = pd.read_csv(results_dir / 'feature_comparison_operating.csv')
budget = pd.read_csv(results_dir / 'feature_comparison_budget.csv')
episodes = pd.read_csv(results_dir / 'episode_coverage.csv')
display(operating.query('split == @split'))
display(budget.query('split == @split'))
display(episodes.query('split == @split'))


## Apriori association rules and source provenance

The rules describe co-occurrence with alerts; they do not establish causal relationships. Source hashes identify local CSV bytes, not the independent authenticity of the approvals.


In [ ]:
display(rules.query('split == @split').head(20))
pd.DataFrame([{'series':name,'sha256':value} for name,value in manifest['input_sha256'].items()])
